# Do You Need a Vector Database for AI Agent Memory? FAISS vs Amazon S3 Vectors

The traveler from [Demo 01](../01-key-value-memory-demo/) is back, now with a season of accumulated memories. They ask:

> *"What should I avoid eating when I go out for dinner on this trip?"*

The answer IS in memory, under the key `dietary_notes`, but the question names no key, and shares no words with the stored note ("eating" vs "vegetarian / shellfish"). That's the dividing line this notebook measures:

| You know... | Use |
|-------------|-----|
| the **key** ("what's my preferred cabin?") | key-value memory (Demo 01): exact, instant, no embeddings |
| only the **meaning** | **vector memory**: embed the memories once, retrieve by similarity |

And once you need vector memory, the real question devs search for: **do you need a vector *database*, or is an in-process index enough?** Two backends, same [Amazon Titan V2](https://docs.aws.amazon.com/bedrock/latest/userguide/titan-embedding-models.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) embeddings, same memories:

- **FAISS**: the index lives in RAM: microsecond queries, zero infrastructure, dies with the process.
- **Amazon S3 Vectors**: the index lives in a vector bucket: you create bucket + index (the notebook self-provisions both), it survives restarts and is reachable from any process.

This demo uses Strands Agents. The pattern carries over to other agent frameworks.

## Install dependencies

In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Configure credentials

- **AWS credentials** (`aws configure`) power both Titan embeddings (Bedrock) and S3 Vectors. The vector bucket + index are **created automatically** if they don't exist.
- **`OPENAI_API_KEY`** is only for the agent conversation at the end (the retrieval measurements need no LLM).

In [2]:
import os

# Bearer-token env vars would override the AWS profile; drop them before boto3 loads.
os.environ.pop('AWS_BEARER_TOKEN', None)
os.environ.pop('AWS_BEARER_TOKEN_BEDROCK', None)

# python-dotenv loads credentials and bucket config from .env
from dotenv import load_dotenv
load_dotenv()

assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env (needed for the agent at the end)'

## The traveler's memories

Ten notes accumulated over past conversations, written through the **same path a live agent uses** (`tools.store_note`), replayed here so the measurement is deterministic. Note that `dietary_notes` shares no words with the question we'll ask.

In [3]:
# memory_stores holds the three stores + the Titan embedder + self-provisioning.
import memory_stores as ms

# tools wires the Strands tools to whichever stores we build here.
import tools

TRAVELER_NOTES = {
    'dietary_notes': 'Vegetarian; severe shellfish allergy, strictly no crustaceans or mollusks.',
    'preferred_cabin': 'Books business class on flights longer than six hours.',
    'airline_status': 'Oneworld Emerald via Iberia Plus; prefers Iberia when fares are close.',
    'layover_rule': 'Refuses overnight connections; anything over four hours is too long.',
    'seat_choice': 'Aisle seat, as far forward as possible, never next to the lavatory.',
    'travel_season': 'Tries to fly shoulder season, late September or early May.',
    'hotel_loyalty': 'No hotel program; picks small independent places near the old town.',
    'packing_habit': 'Carry-on only, no matter the trip length.',
    'budget_ceiling': 'Keeps round-trip fares under 1,500 USD unless it is a special occasion.',
    'airport_home': 'Based near JFK; can use EWR if the fare difference is over 200 USD.',
}

QUESTION = 'What should I avoid eating when I go out for dinner on this trip?'

kv = ms.KeyValueStore()
faiss_store = ms.FaissStore()
s3v = ms.S3VectorStore()   # self-provisions the vector bucket + index if missing
s3v.clear()                # rerun-safe: drop this demo's old vectors

tools.init_stores(kv, faiss_store)
for key, note in TRAVELER_NOTES.items():
    tools.store_note(key, note)          # key-value + FAISS
    s3v.put(key, note, ms.embed(note))   # S3 Vectors (same Titan embedding)

print(f'stored {len(TRAVELER_NOTES)} memories in all three stores')

stored 10 memories in all three stores


---
## Test 1. The key-value limit

The best a key-value store can do without a key: scan for shared words, or dump everything into context. Watch both fail the semantic question: the scan misses (no shared words), and dump-all means paying for the whole memory on every question.

In [4]:
hits = kv.keyword_search(QUESTION)
found = any('shellfish' in h for h in hits)
print(f'keyword scan hits: {len(hits)} (answer found: {found})')
for h in hits:
    print('  -', h[:70])

dump = kv.dump_all()
print(f'\nfallback dump-all: {len(dump):,} chars of memory into context, every question')

keyword scan hits: 4 (answer found: False)
  - Oneworld Emerald via Iberia Plus; prefers Iberia when fares are close.
  - Tries to fly shoulder season, late September or early May.
  - Carry-on only, no matter the trip length.
  - Keeps round-trip fares under 1,500 USD unless it is a special occasion

fallback dump-all: 648 chars of memory into context, every question


---
## Test 2. FAISS: retrieval by meaning, in-process

Same memories, embedded once with Titan V2. The question embeds to a vector; cosine similarity finds `dietary_notes` even though not one meaningful word overlaps. FAISS answers in **microseconds**: it's a RAM index.

In [5]:
qvec, embed_ms = ms.timed(ms.embed, QUESTION)
faiss_store.query(qvec, 3)   # warm-up: first call pays one-time setup
hits, query_ms = ms.timed(faiss_store.query, qvec, 3)

for text, score in hits:
    print(f'  {score:.3f}  {text[:70]}')
print(f'\nembed: {embed_ms:.0f} ms | FAISS query: {query_ms:.2f} ms')

  0.231  Vegetarian; severe shellfish allergy, strictly no crustaceans or moll
  0.148  Carry-on only, no matter the trip length.
  0.118  Keeps round-trip fares under 1,500 USD unless it is a special occasion

embed: 512 ms | FAISS query: 0.06 ms


---
## Test 3. Amazon S3 Vectors: the index that survives

Same memories, same embeddings, but the index lives in a **vector bucket**, not in RAM. The query is a network call (`query_vectors`), so it costs milliseconds instead of microseconds. What you buy: **persistence**. A brand-new client object (nothing carried over in memory, the "restart") still sees every vector.

In [6]:
s3v.query(qvec, 3)           # warm-up: first call pays TLS/connection setup
hits, query_ms = ms.timed(s3v.query, qvec, 3)

for text, score in hits:
    print(f'  {score:.3f}  {text[:70]}')
print(f'\nS3 Vectors query: {query_ms:.0f} ms')

fresh = ms.S3VectorStore()   # new client, nothing in RAM: the 'restart'
print(f"fresh client sees {fresh.count()}/{len(TRAVELER_NOTES)} vectors: index survived")

  0.231  Vegetarian; severe shellfish allergy, strictly no crustaceans or moll
  0.148  Carry-on only, no matter the trip length.
  0.118  Keeps round-trip fares under 1,500 USD unless it is a special occasion

S3 Vectors query: 175 ms


fresh client sees 10/10 vectors: index survived


---
## Test 4. A real agent choosing between the tools

The agent gets both recall tools. Their docstrings say when each applies (key lookup for known identifiers, semantic for meaning), so the agent picks per question without prompt engineering.

In [7]:
# OTEL_SDK_DISABLED silences OpenTelemetry tracing noise in notebook output.
os.environ['OTEL_SDK_DISABLED'] = 'true'

# Agent is the Strands agent loop: model + tools until the answer is done.
from strands import Agent

# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands.models.openai import OpenAIModel

MODEL = OpenAIModel(model_id='gpt-4o-mini')

# Amazon Bedrock instead (no OpenAI key; uses your AWS credentials):
# from strands.models import BedrockModel
# MODEL = BedrockModel(model_id='openai.gpt-oss-120b-1:0', region_name='us-west-2')

agent = Agent(
    model=MODEL,
    system_prompt=('You are a travel assistant with long-term memory of this user. '
                   'Be concise: 2-3 sentences maximum.'),
    tools=[tools.recall_by_key, tools.recall_semantic, tools.remember_note],
    callback_handler=None,
)

resp = agent(QUESTION)
print(f'User: {QUESTION}\nAgent: {str(resp).strip()}')

resp = agent('And what is my preferred cabin?')
print(f'\nUser: And what is my preferred cabin?\nAgent: {str(resp).strip()}')

User: What should I avoid eating when I go out for dinner on this trip?
Agent: You should avoid eating shellfish, particularly crustaceans and mollusks, due to your severe allergy.



User: And what is my preferred cabin?
Agent: Your preferred cabin is business class for flights longer than six hours.


---
## The decision table

| You need | Pick | Why |
|----------|------|-----|
| Facts under known keys (profile, prefs) | **Key-value** (Demo 01) | Exact and instant. Don't pay embeddings for lookups |
| Search by meaning, single process, prototype | **FAISS** | Microsecond queries, zero infrastructure. Gone when the process dies |
| Search by meaning, persistent, shared | **Amazon S3 Vectors** | Milliseconds per query, nothing to administer, survives restarts, any process can read it |
| Multi-hop questions over relationships | **Graph** → [Demo 03](../03-graph-memory-demo/) | Similarity can't follow edges |

Two honest footnotes from the measurements: **the embedding call dominates** (~0.5 s per question with Titan V2, the same cost for both backends), and for 10 memories a dump-all is still cheap; vector memory earns its keep as memory GROWS (hundreds of notes, where dump-all costs thousands of tokens per question).

**Production note:** for multi-tenant SaaS memory on S3 Vectors, see the [`strands-s3-vectors-memory`](https://github.com/aws-samples/data-for-saas-patterns/tree/main/samples/multi-tenant-strands-s3-vectors-memory) community plugin (one index per tenant with IAM-scoped credentials).

---
## Cleanup: delete AWS resources (optional)

Run the cell below only if you want to remove the S3 Vectors bucket and index created by this notebook. The self-provisioning code re-creates them the next time you run a test, so cleanup is purely optional.

In [ ]:
import boto3, os
from dotenv import load_dotenv
load_dotenv()

AWS_REGION = os.getenv('AWS_REGION', 'us-east-1')
profile = os.getenv('AWS_PROFILE')
session = boto3.Session(profile_name=profile) if profile else boto3.Session()

bucket = os.getenv('VECTOR_BUCKET', 'agent-memory-demo-vectors')
index  = os.getenv('VECTOR_INDEX', 'traveler-memories')

s3v = session.client('s3vectors', region_name=AWS_REGION)

# Delete the vectors and index this demo created.
try:
    resp = s3v.list_vectors(vectorBucketName=bucket, indexName=index)
    keys = [v['key'] for v in resp.get('vectors', [])]
    if keys:
        s3v.delete_vectors(vectorBucketName=bucket, indexName=index, keys=keys)
        print(f"  deleted {len(keys)} vector(s) from index '{index}'")
    s3v.delete_index(vectorBucketName=bucket, indexName=index)
    print(f"  deleted index '{index}'")
except s3v.exceptions.NotFoundException:
    print(f"  index '{index}' not found (already deleted or never created)")

# Only delete the bucket if it is now empty (other demos may share it).
try:
    remaining = s3v.list_indexes(vectorBucketName=bucket).get('indexes', [])
    if remaining:
        names = [i['indexName'] for i in remaining]
        print(f"  bucket '{bucket}' still has indexes {names}, leaving it (other demos use it)")
    else:
        s3v.delete_vector_bucket(vectorBucketName=bucket)
        print(f"  deleted S3 Vectors bucket '{bucket}' (was empty)")
except s3v.exceptions.NotFoundException:
    print(f"  bucket '{bucket}' not found (already deleted or never created)")